# Accounts overview

Lists every account in the active data repo with its opening balance and (if applicable) the most recent rate that the scenario applies to it.

Set `DATA_DIR` and `SCENARIO` below to point at the repo / scenario you want to inspect.

In [1]:
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", "../data/shared/sim"))
SCENARIO = os.environ.get("ANALYSIS_SCENARIO")  # None -> auto-detect single scenario
AS_OF = None  # e.g. '2026-05-31' to pin rate lookups
DATA_DIR

PosixPath('../data/shared/sim')

## Rich table

In [2]:
from rich.console import Console
from analysis.sim.accounts import load_accounts_view, render_accounts_table

rows = load_accounts_view(DATA_DIR, SCENARIO, as_of=AS_OF)
Console().print(render_accounts_table(rows, title=f"accounts ({DATA_DIR.name})"))

                                                  accounts (sim)                                                   
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Name           ┃ Kind      ┃ Subkind         ┃ Owner ┃ Tax          ┃ Opening ┃  Rate ┃ Series         ┃ Role   ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Business       │ asset     │ checking        │ joint │ taxable      │         │ 4.50% │ checking       │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ Family WF PMA  │ asset     │ checking        │ joint │ taxable      │         │ 4.50% │ checking       │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ Hiromi FECU    │ asset     │ checking        │ ho    │ taxable      │         │ 4.50% │ checking       │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ Hiromi Card WF │ asset     │ checking        │ ho    │ taxable      │         │ 4.50% │ checking       │ yield  │
│ HC             │           │                 │       │              │         │       │ current        │        │
│ WF CA          │ asset     │ checking        │ joint │ taxable      │         │ 4.50% │ checking       │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ WF CM          │ asset     │ checking        │ joint │ taxable      │         │ 4.50% │ checking       │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ Bob FECU First │ asset     │ savings         │ rwk   │ taxable      │         │ 4.50% │ savings        │ yield  │
│ 500            │           │                 │       │              │         │       │ current        │        │
│ Bob HSA Cash   │ asset     │ hsa             │ rwk   │ tax_free     │         │       │                │        │
│ Hiromi FECU    │ asset     │ savings         │ ho    │ taxable      │         │ 4.50% │ savings        │ yield  │
│ First 500      │           │                 │       │              │         │       │ current        │        │
│ Hiromi FECU    │ asset     │ savings         │ ho    │ taxable      │         │ 4.50% │ money market   │ yield  │
│ Money Market   │           │                 │       │              │         │       │ current        │        │
│ Hiromi HSA     │ asset     │ hsa             │ ho    │ tax_free     │         │ 6.20% │ mutual fund    │ growth │
│                │           │                 │       │              │         │       │ retirement     │        │
│ Hiromi WF MM   │ asset     │ savings         │ ho    │ taxable      │         │ 4.50% │ money market   │ yield  │
│                │           │                 │       │              │         │       │ current        │        │
│ Bob HSA        │ asset     │ hsa             │ rwk   │ tax_free     │         │ 6.20% │ mutual fund    │ growth │
│ Invested       │           │                 │       │              │         │       │ retirement     │        │
│ Bob Vanguard   │ asset     │ brokerage_taxab │ rwk   │ taxable      │         │ 6.20% │ mutual fund    │ growth │
│                │           │ le              │       │              │         │       │ retirement     │        │
│ Hiromi         │ asset     │ brokerage_taxab │ ho    │ taxable      │         │ 6.20% │ mutual fund    │ growth │
│ Fidelity       │           │ le              │       │              │         │       │ retirement     │        │
│ Bob FECU       │ asset     │ traditional_ira │ rwk   │ tax_deferred │         │ 6.20% │ mutual fund    │ growth │
│                │           │                 │       │

## DataFrame view

Same data as a pandas DataFrame for filtering, grouping, or export.

In [3]:
from analysis.sim.accounts import accounts_dataframe

df = accounts_dataframe(DATA_DIR, SCENARIO, as_of=AS_OF)
df

,name,kind,subkind,owner,tax_treatment,opening_balance,rate_role,rate_series,annual_rate_pct,opened,closed
0,Business,asset,checking,joint,taxable,None,yield,checking current,4.5,,
1,Family WF PMA,asset,checking,joint,taxable,None,yield,checking current,4.5,,
2,Hiromi FECU,asset,checking,ho,taxable,None,yield,checking current,4.5,,
3,Hiromi Card WF HC,asset,checking,ho,taxable,None,yield,checking current,4.5,,
4,WF CA,asset,checking,joint,taxable,None,yield,checking current,4.5,,
5,WF CM,asset,checking,joint,taxable,None,yield,checking current,4.5,,
6,Bob FECU First 500,asset,savings,rwk,taxable,None,yield,savings current,4.5,,
7,Bob HSA Cash,asset,hsa,rwk,tax_free,None,NaN,NaN,NaN,,
8,Hiromi FECU First 500,asset,savings,ho,taxable,None,yield,savings current,4.5,,
9,Hiromi FECU Money Market,asset,savings,ho,taxable,None,yield,money market current,4.5,,


In [4]:
# Summary: total opening balance per kind.
if not df.empty:
    summary = (
        df.dropna(subset=["opening_balance"])
          .groupby("kind", as_index=False)["opening_balance"]
          .sum()
          .sort_values("opening_balance", ascending=False)
    )
    display(summary)

,kind,opening_balance


## Balance history (latest run)

Pick one or more accounts (Cmd/Ctrl-click for multi-select) to plot their
month-end balances from the most recent run under
`data/shared/state/runs/`.

In [5]:
import sqlite3
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = REPO / "data" / "shared" / "state" / "runs"

run_files = sorted(RUNS_DIR.glob("*.sqlite3"))
if not run_files:
    raise FileNotFoundError(f"no run files under {RUNS_DIR}")
RUN_FILE = run_files[-1]
print(f"latest run: {RUN_FILE.name}")

run_con = sqlite3.connect(f"file:{RUN_FILE}?mode=ro", uri=True)

balances = pd.read_sql(
    "SELECT s.month, a.name AS account, a.kind, ast.balance "
    "FROM account_state ast "
    "JOIN simulation_step s ON s.id = ast.step_id "
    "JOIN account a         ON a.id = ast.account_id "
    "ORDER BY s.month, a.name",
    run_con,
)
balances["month"] = pd.to_datetime(balances["month"])
balances["balance"] = balances["balance"] / 100  # cents -> dollars

balance_pivot = balances.pivot_table(
    index="month", columns="account", values="balance", aggfunc="last"
)
account_kinds = balances.drop_duplicates("account").set_index("account")["kind"]
print(f"{balance_pivot.shape[1]} accounts, {len(balance_pivot)} months")

latest run: 2026-05-21T193104Z_current_46f6f10.sqlite3
53 accounts, 416 months


In [6]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

_PLOTTABLE_KINDS = {"asset", "liability"}
_plottable = [
    name for name in balance_pivot.columns
    if account_kinds.get(name) in _PLOTTABLE_KINDS
]
# Sort by latest balance, largest first, for convenient picking.
_plottable.sort(
    key=lambda n: abs(balance_pivot[n].iloc[-1] or 0), reverse=True,
)

picker = widgets.SelectMultiple(
    options=_plottable,
    value=tuple(_plottable[:3]),
    rows=min(15, max(5, len(_plottable))),
    description="Accounts:",
    layout=widgets.Layout(width="50%"),
)
out = widgets.Output()

def _redraw(change=None):
    out.clear_output(wait=True)
    with out:
        selected = list(picker.value)
        if not selected:
            print("(pick one or more accounts)")
            return
        fig, ax = plt.subplots(figsize=(10, 5))
        for name in selected:
            ax.plot(balance_pivot.index, balance_pivot[name], label=name)
        ax.set_title(f"Account balances — {RUN_FILE.stem}")
        ax.set_ylabel("USD")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize="small")
        fig.autofmt_xdate()
        plt.show()

picker.observe(_redraw, names="value")
display(picker, out)
_redraw()

SelectMultiple(description='Accounts:', index=(0, 1, 2), layout=Layout(width='50%'), options=('Hiromi Vanguard…

Output()